# 02.5 — Re-scrape STEM modules (fix for the missing Maths / Engineering / Computing / Physics data)

## What was wrong with the v1 scrape

The Module Bank has **two different page layouts** behind the same URL:

| Layout | Triggered by | Used by |
|---|---|---|
| Old (`sys=0`) | Default in v1's notebook 02 | HASS, Business, Bio, Geo, Law, Psychology, Education, etc. |
| New (`sys=1`) | Has to be requested explicitly | **Maths (MTH), Computing (ECM), Engineering (ENG), Physics (PHY)** |

When the v1 scraper hit a STEM module with `sys=0`, the server returned a near-empty page (`tables = 0`). The fetch "succeeded" (HTTP 200), so `fetch_ok = True`, but every field came back NaN. That silently dropped ~850 STEM modules from the analysis.

## What this notebook does

1. Read the existing `modules_details.csv`.
2. Find every row where `fetch_ok = True` but the assessment percentages are all NaN — those are the broken ones.
3. Re-fetch each one with `sys=1` and parse the new page layout (which uses label-value-label-value rows in tables instead of H2/H3-and-table sections).
4. Back up the original file to `modules_details_v1backup.csv` and write the corrected version back to `modules_details.csv`. **Notebook 03 onwards then works without any further changes.**

Roughly 850 rows need re-fetching, with a 1 s gap between requests = ~14 minutes.

## 1. Setup

In [9]:
import re
import shutil
import time
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup

HERE = Path.cwd()
PROJECT_ROOT = HERE if (HERE / 'data' / 'raw').exists() else HERE.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'module_bank'
DETAILS_PATH = RAW_DIR / 'modules_details.csv'
BACKUP_PATH  = RAW_DIR / 'modules_details_v1backup.csv'

DETAIL_URL = 'https://www.exeter.ac.uk/study/studyinformation/modules/info'
HEADERS = {'User-Agent': 'Mozilla/5.0 (Exeter BEE2041 student project; educational research)'}
SLEEP = 1.0
TIMEOUT = 20

df = pd.read_csv(DETAILS_PATH)
print(f'Loaded modules_details.csv: {len(df):,} rows')

Loaded modules_details.csv: 9,073 rows


## 2. Find the broken rows

Broken rows have `fetch_ok = True` (the page was served) but **every** assessment percentage is missing. That's the unmistakable signature of the empty-page-from-`sys=0` problem.

In [11]:
assess_cols = ['coursework_pct', 'written_exam_pct', 'practical_exam_pct']

broken = df[(df['fetch_ok'] == True) & df[assess_cols].isna().all(axis=1)].copy()
print(f'Broken rows: {len(broken):,}')

# Show which depts are affected.
broken['dept_code'] = broken['module_code'].astype(str).str.extract(r'^([A-Z]{3})')
print('\nBy dept:')
print(broken['dept_code'].value_counts())

Broken rows: 1,549

By dept:
dept_code
MTH    471
ENG    431
ECM    376
PHY    173
PSY     13
EDU     11
EMP     11
BEM      8
BIO      7
LAW      7
BEP      5
GEO      4
SOC      4
CLA      4
BEE      4
DRA      3
EFP      3
ARC      3
EAF      2
HIH      2
CMM      2
PYC      2
BUS      1
POL      1
EAS      1
Name: count, dtype: int64


## 3. Parser for the *new* (`sys=1`) page layout

The new layout puts everything in tables — no H2/H3 headings to anchor to. Each table is identified by **what its first row says**.

Pattern of the rows we care about:
- Meta:        `MODULE TITLE | <title> | CREDIT VALUE | <n>`
- Cohort:      `Number of Students Taking Module (anticipated) | <n>`
- Hours:       `Scheduled Learning ... | <n> | Guided Independent Study | <n> | Placement / Study Abroad | <n>`
- Assessment:  `Coursework | <pct> | Written Exams | <pct> | Practical Exams | <pct>`
- Details:     header row contains `Form of Assessment` AND `% of Credit`; data rows = items.
- NQF:         `NQF LEVEL (FHEQ) | <n> | AVAILABLE AS DISTANCE LEARNING | <Yes/No>`

In [13]:
def _clean(s):
    return re.sub(r'\s+', ' ', s or '').strip()

def _int_or_none(s):
    if s is None: return None
    m = re.search(r'-?\d+', str(s))
    return int(m.group(0)) if m else None

def _row_cells(tr):
    return [_clean(c.get_text(' ', strip=True)) for c in tr.find_all(['th', 'td'])]

def parse_new_page(html):
    """Parse a sys=1 module descriptor page.  Returns the same dict shape as v1."""
    soup = BeautifulSoup(html, 'html.parser')
    main = soup.find(id='main-col') or soup.find('main') or soup

    out = {k: None for k in [
        'module_title', 'credits', 'num_students_anticipated',
        'scheduled_hours', 'independent_hours', 'placement_hours',
        'coursework_pct', 'written_exam_pct', 'practical_exam_pct',
        'n_summative_items', 'nqf_level', 'distance_learning',
        'origin_date', 'last_revision_date',
    ]}

    # Module title comes from the H1 in the new layout.
    h1 = main.find('h1')
    if h1:
        title = _clean(h1.get_text(' ', strip=True))
        out['module_title'] = re.sub(r'\s*-\s*\d{4}\s+entry\s*$', '', title)

    for t in main.find_all('table'):
        first = t.find('tr')
        if not first:
            continue
        cells = _row_cells(first)
        if not cells:                         # empty header row -> skip
            continue
        labels = [c.lower() for c in cells]
        first_label = labels[0]               # safe now: we've checked cells is non-empty

        # --- Meta row: MODULE TITLE | x | CREDIT VALUE | y ---
        if len(cells) >= 4 and 'module title' in first_label and 'credit' in labels[2]:
            if not out['module_title']:
                out['module_title'] = cells[1]
            out['credits'] = _int_or_none(cells[3])

        # --- Cohort: Number of Students Taking Module (anticipated) | n ---
        if len(cells) >= 2 and 'students taking module' in first_label:
            out['num_students_anticipated'] = _int_or_none(cells[1])

        # --- Hours block ---
        if len(cells) >= 4 and 'scheduled' in first_label:
            out['scheduled_hours'] = _int_or_none(cells[1])
            if 'independent' in labels[2]:
                out['independent_hours'] = _int_or_none(cells[3])
            if len(cells) >= 6 and ('placement' in labels[4] or 'study abroad' in labels[4]):
                out['placement_hours'] = _int_or_none(cells[5])

        # --- Assessment %: Coursework | x | Written Exams | y | Practical Exams | z ---
        if (len(cells) >= 6
            and 'coursework' in first_label
            and 'written' in labels[2]
            and 'practical' in labels[4]):
            out['coursework_pct']     = _int_or_none(cells[1])
            out['written_exam_pct']   = _int_or_none(cells[3])
            out['practical_exam_pct'] = _int_or_none(cells[5])

        # --- Details of summative assessment: header has both 'Form of Assessment' and '% of Credit' ---
        if ('form of assessment' in first_label
            and any('% of credit' in lab for lab in labels)):
            data_rows = t.find_all('tr')[1:]
            n_items = sum(
                1 for tr in data_rows
                if tr.find(['th', 'td']) and _clean(tr.find(['th', 'td']).get_text(' ', strip=True))
            )
            if n_items > 0:
                out['n_summative_items'] = n_items

        # --- NQF level + distance learning ---
        if len(cells) >= 2 and 'nqf level' in first_label:
            out['nqf_level'] = _int_or_none(cells[1])
            if len(cells) >= 4 and 'distance learning' in labels[2]:
                out['distance_learning'] = cells[3]

    return out

# Quick sanity check on a real STEM module.
_r = requests.get(DETAIL_URL, params={'moduleCode': 'MTH1001', 'ay': '2024/5', 'sys': '1'},
                  headers=HEADERS, timeout=TIMEOUT)
parse_new_page(_r.text)

{'module_title': 'Mathematical Structures',
 'credits': 30,
 'num_students_anticipated': 270,
 'scheduled_hours': 76,
 'independent_hours': 224,
 'placement_hours': None,
 'coursework_pct': 20,
 'written_exam_pct': 80,
 'practical_exam_pct': 0,
 'n_summative_items': 4,
 'nqf_level': 4,
 'distance_learning': 'No',
 'origin_date': None,
 'last_revision_date': None}

## 4. Re-fetch all the broken rows with `sys=1`

We collect the new field values into a dict keyed by `(module_code, academic_year)` and apply them to `df` after the loop. Saves progress to `stem_corrections.csv` every 50 rows so an interrupted run doesn't lose work.

In [15]:
CORRECTIONS_PATH = RAW_DIR / 'stem_corrections.csv'
FIELDS = ['module_title', 'credits', 'num_students_anticipated',
         'scheduled_hours', 'independent_hours', 'placement_hours',
         'coursework_pct', 'written_exam_pct', 'practical_exam_pct',
         'n_summative_items', 'nqf_level', 'distance_learning']

# Resume if a previous run was interrupted.
if CORRECTIONS_PATH.exists():
    done = pd.read_csv(CORRECTIONS_PATH)
    done_keys = set(zip(done['module_code'], done['academic_year']))
    print(f'Resuming: {len(done):,} rows already corrected.')
else:
    done = pd.DataFrame(columns=['module_code', 'academic_year'] + FIELDS)
    done_keys = set()

todo = broken[~broken.apply(
    lambda r: (r['module_code'], r['academic_year']) in done_keys, axis=1
)].reset_index(drop=True)
print(f'Still to re-fetch: {len(todo):,}')

rows_buffer = []
for i, row in todo.iterrows():
    code, year = row['module_code'], row['academic_year']
    try:
        r = requests.get(DETAIL_URL,
                         params={'moduleCode': code, 'ay': year, 'sys': '1'},
                         headers=HEADERS, timeout=TIMEOUT)
        parsed = parse_new_page(r.text) if r.status_code == 200 else {f: None for f in FIELDS}
    except requests.RequestException as e:
        print(f'  ! network error {code} {year}: {e}')
        parsed = {f: None for f in FIELDS}

    rows_buffer.append({'module_code': code, 'academic_year': year, **{f: parsed.get(f) for f in FIELDS}})

    if (i + 1) % 50 == 0 or (i + 1) == len(todo):
        done = pd.concat([done, pd.DataFrame(rows_buffer)], ignore_index=True)
        done.to_csv(CORRECTIONS_PATH, index=False)
        rows_buffer = []
        recovered = done['coursework_pct'].notna().sum()
        print(f'  {i + 1:5d} / {len(todo):5d}  last: {code} {year}  cw={parsed["coursework_pct"]}  recovered_so_far={recovered}')
    time.sleep(SLEEP)

print(f'\nDone. {len(done):,} corrections written to {CORRECTIONS_PATH.name}')
print(f'Of those, {done["coursework_pct"].notna().sum():,} now have valid assessment data.')

Still to re-fetch: 1,549


C:\Users\kenna\AppData\Local\Temp\ipykernel_5032\2632823659.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  done = pd.concat([done, pd.DataFrame(rows_buffer)], ignore_index=True)


     50 /  1549  last: ECMM103 2019/0  cw=100  recovered_so_far=50
    100 /  1549  last: MTH2004 2019/0  cw=0  recovered_so_far=98
    150 /  1549  last: ARCM007 2019/0  cw=None  recovered_so_far=146
    200 /  1549  last: ENG3008A 2020/1  cw=None  recovered_so_far=190
    250 /  1549  last: ENGM006 2020/1  cw=30  recovered_so_far=233
    300 /  1549  last: ECMM162 2020/1  cw=0  recovered_so_far=281
    350 /  1549  last: MTH3040 2020/1  cw=20  recovered_so_far=329
    400 /  1549  last: ENG1009 2021/2  cw=45  recovered_so_far=373
    450 /  1549  last: ENG3010 2021/2  cw=0  recovered_so_far=419
    500 /  1549  last: ENGM010 2021/2  cw=100  recovered_so_far=463
    550 /  1549  last: ECMM180 2021/2  cw=100  recovered_so_far=511
    600 /  1549  last: MTHM007 2021/2  cw=100  recovered_so_far=559
    650 /  1549  last: ECM1110 2022/3  cw=20  recovered_so_far=601
    700 /  1549  last: ENG3207 2022/3  cw=30  recovered_so_far=651
    750 /  1549  last: ECMM108 2022/3  cw=50  recovered_so

## 5. Merge the corrections back into `modules_details.csv`

Original is backed up to `modules_details_v1backup.csv` first, then overwritten with the corrected rows merged in. Subsequent notebooks (03+) can read `modules_details.csv` as before.

In [17]:
if not BACKUP_PATH.exists():
    shutil.copy2(DETAILS_PATH, BACKUP_PATH)
    print(f'Backed up original to: {BACKUP_PATH.name}')
else:
    print(f'Backup already exists: {BACKUP_PATH.name} (left untouched)')

corrections = pd.read_csv(CORRECTIONS_PATH)
fixed = corrections[corrections['coursework_pct'].notna()].copy()
print(f'Applying {len(fixed):,} corrections...')

# Use module_code+academic_year as the key.
df_indexed   = df.set_index(['module_code', 'academic_year'])
fixed_indexed = fixed.set_index(['module_code', 'academic_year'])

for col in FIELDS:
    if col in fixed_indexed.columns:
        # Only update where the original was NaN — never overwrite real data.
        mask_overlap = df_indexed.index.isin(fixed_indexed.index)
        for key in fixed_indexed.index:
            if key in df_indexed.index and pd.isna(df_indexed.at[key, col]):
                v = fixed_indexed.at[key, col]
                if pd.notna(v):
                    df_indexed.at[key, col] = v

df_out = df_indexed.reset_index()
df_out.to_csv(DETAILS_PATH, index=False)
print(f'Wrote corrected modules_details.csv: {len(df_out):,} rows')

# Summary of what changed.
print('\nAssessment-data completeness, before vs after:')
before_pct = df['coursework_pct'].notna().mean() * 100
after_pct  = df_out['coursework_pct'].notna().mean() * 100
print(f'  coursework_pct non-null: {before_pct:.1f}%  →  {after_pct:.1f}%')
print(f'  Rows newly populated: {(df_out["coursework_pct"].notna() & df["coursework_pct"].isna()).sum():,}')

Backed up original to: modules_details_v1backup.csv
Applying 1,431 corrections...
Wrote corrected modules_details.csv: 9,073 rows

Assessment-data completeness, before vs after:
  coursework_pct non-null: 82.7%  →  98.5%
  Rows newly populated: 1,431


## Next step

Re-run **notebooks 03 → 04 → 05 → 06**. Notebook 03 will now find UG modules in MTH, ECM, ENG, PHY and propagate them through the pipeline. Combined with the PSY mapping fix in notebook 04, the model should grow from 54 rows to roughly 80–100.